In [ ]:
# ── Step 1: Run this cell first ─────────────────────────────────────────────
# After it finishes → Runtime → Restart session → Runtime → Run all
!pip install -q faker
print('✅ Done. Now: Runtime → Restart session, then Runtime → Run all')

# Philippine National Multi-Disease Vaccination Campaign
## 03 — Dataset Generator
**Synthetic Dataset | 2024–2028 Longitudinal | EIF Cohort 10 — Eskwelabs**

---
> **Usage:** Run all cells top-to-bottom. Change `RANDOM_SEED` in Cell 1 to produce a  
> different but equally valid dataset — mimicking natural variance in real campaign data.  
> Outputs are saved to `data/observable/` (Silver) and `data/gold/` (Gold) as CSV files.

| Section | Description |
|---|---|
| 1. Modular Constants | Global parameters shared by all engines |
| 2. Engine A — Population | `dim_people` generator |
| 3. Engine B — Sites | `dim_sites` generator |
| 4. Engine C — Appointments | `fact_appointments` with behavioral no-show model |
| 5. Engine D — Cold Chain | `fact_inventory_shipments` with Arrhenius model |
| 6. Engine E — Vaccination Events | `fact_vaccination_events` with efficacy & AEFI |
| 7. Engine F — Gold Layer | Derived analytical tables |
| 8. Master Factory | Orchestration cell — runs full simulation & saves CSVs |

---
## 1. Modular Constants

**Description:** Defines all global simulation parameters used by every engine below.  
Modify `RANDOM_SEED` to regenerate with natural variance.

In [ ]:
import os, uuid, math, warnings
import numpy as np
import pandas as pd
from datetime import date, timedelta, datetime
from faker import Faker
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_SEED       = 42          # Change to regenerate with natural variance
np.random.seed(RANDOM_SEED)
fake = Faker(['en_PH', 'fil_PH'])
Faker.seed(RANDOM_SEED)

# ── Simulation scope ─────────────────────────────────────────────────────────
START_DATE        = date(2024, 1, 1)
END_DATE          = date(2028, 12, 31)
N_PEOPLE          = 5_000
N_SITES           = 150
DATA_ROOT         = 'data'
SUBFOLDERS        = ['observable', 'gold']

# ── Behavioral model coefficients (logistic no-show model) ───────────────────
# Calibrated against DOH 2023 EPI coverage survey so baseline no-show ≈ 16%.
BETA = {
    'intercept'    : -2.55,   # baseline log-odds of no-show
    'distance'     :  0.06,   # +distance to site  → +no-show
    'sec'          :  0.22,   # +SEC score (D/E)   → +no-show (access barrier)
    'dose_num'     :  0.20,   # later doses in a series → mild +no-show (fatigue)
    'prior_noshow' :  0.70,   # history of no-show → strong +no-show
    'rurality'     :  0.35,   # +rurality          → +no-show
}

# ── Causal treatment: reminder_sent (SMS/call before the appointment) ─────────
# The reminder LOWERS no-show log-odds, and does so MORE for high-barrier patients
# (heterogeneous treatment effect → non-trivial CATE). Effect size:
#     Δlog-odds = −(REMINDER_BASE + REMINDER_HETERO · barrier_score)
# where barrier_score ∈ [0,1] summarizes structural access difficulty.
REMINDER_BASE   = 0.30    # baseline reminder effect (everyone benefits a little)
REMINDER_HETERO = 1.30    # extra benefit scaled by the patient's access barrier

# ── Philippine regional population weights (PSA 2024) ────────────────────────
REGIONAL_POP_WEIGHTS = {
    'NCR': 0.1430, 'CAR': 0.0175, 'Region I': 0.0520,
    'Region II': 0.0380, 'Region III': 0.1100, 'Region IV-A': 0.1580,
    'MIMAROPA': 0.0310, 'Region V': 0.0580, 'Region VI': 0.0760,
    'Region VII': 0.0830, 'Region VIII': 0.0440, 'Region IX': 0.0390,
    'Region X': 0.0510, 'Region XI': 0.0630, 'Region XII': 0.0510,
    'CARAGA': 0.0240, 'BARMM': 0.0420,
}
REGIONS = list(REGIONAL_POP_WEIGHTS.keys())
# Normalize weights — must sum exactly to 1.0 for numpy
_rw = np.array(list(REGIONAL_POP_WEIGHTS.values()))
REGION_WEIGHTS = (_rw / _rw.sum()).tolist()

# ── Province lookup per region ───────────────────────────────────────────────
REGION_PROVINCES = {
    'NCR': ['Metro Manila'], 'CAR': ['Benguet','Ifugao','Mountain Province','Kalinga','Apayao','Abra'],
    'Region I': ['Ilocos Norte','Ilocos Sur','La Union','Pangasinan'],
    'Region II': ['Cagayan','Isabela','Nueva Vizcaya','Quirino','Batanes'],
    'Region III': ['Aurora','Bataan','Bulacan','Nueva Ecija','Pampanga','Tarlac','Zambales'],
    'Region IV-A': ['Batangas','Cavite','Laguna','Quezon','Rizal'],
    'MIMAROPA': ['Marinduque','Occidental Mindoro','Oriental Mindoro','Palawan','Romblon'],
    'Region V': ['Albay','Camarines Norte','Camarines Sur','Catanduanes','Masbate','Sorsogon'],
    'Region VI': ['Aklan','Antique','Capiz','Guimaras','Iloilo','Negros Occidental'],
    'Region VII': ['Bohol','Cebu','Negros Oriental','Siquijor'],
    'Region VIII': ['Biliran','Eastern Samar','Leyte','Northern Samar','Samar','Southern Leyte'],
    'Region IX': ['Zamboanga del Norte','Zamboanga del Sur','Zamboanga Sibugay'],
    'Region X': ['Bukidnon','Camiguin','Lanao del Norte','Misamis Occidental','Misamis Oriental'],
    'Region XI': ['Compostela Valley','Davao del Norte','Davao del Sur','Davao Occidental','Davao Oriental'],
    'Region XII': ['Cotabato','Sarangani','South Cotabato','Sultan Kudarat'],
    'CARAGA': ['Agusan del Norte','Agusan del Sur','Dinagat Islands','Surigao del Norte','Surigao del Sur'],
    'BARMM': ['Basilan','Lanao del Sur','Maguindanao','Sulu','Tawi-Tawi'],
}

# ── Rurality index per region (0=urban, 1=remote rural) ─────────────────────
REGION_RURALITY = {
    'NCR': 0.05, 'CAR': 0.72, 'Region I': 0.45, 'Region II': 0.60,
    'Region III': 0.30, 'Region IV-A': 0.28, 'MIMAROPA': 0.70,
    'Region V': 0.62, 'Region VI': 0.50, 'Region VII': 0.40,
    'Region VIII': 0.65, 'Region IX': 0.68, 'Region X': 0.55,
    'Region XI': 0.48, 'Region XII': 0.58, 'CARAGA': 0.71, 'BARMM': 0.78,
}

# ── Vaccine configurations ───────────────────────────────────────────────────
VACCINE_CONFIGS = {
    'COVID_Booster': {
        'doses': 1, 'interval_days': None,
        'E_peak': 0.92, 'phi': 0.60, 'lambda_f': 0.0050, 'lambda_s': 0.0003,
        'R0': 5.0, 'p_HIT': 0.80, 'AEFI_per_1000': 85.0,
        'booster_threshold': 0.50, 'annual': False,
        'age_targets': ['18-59', '60+'],
        'unit_cost_php': 850,
        'shelf_life_days': 270,
        'route': 'IM', 'site_of_admin': 'Deltoid',
    },
    'Influenza': {
        'doses': 1, 'interval_days': 365,
        'E_peak': 0.70, 'phi': 0.80, 'lambda_f': 0.0060, 'lambda_s': 0.0010,
        'R0': 1.5, 'p_HIT': 0.33, 'AEFI_per_1000': 12.0,
        'booster_threshold': None, 'annual': True,
        'age_targets': ['0-5', '18-59', '60+'],
        'unit_cost_php': 450,
        'shelf_life_days': 180,
        'route': 'IM', 'site_of_admin': 'Deltoid',
    },
    'MMR': {
        'doses': 2, 'interval_days': 28,
        'E_peak': 0.97, 'phi': 0.10, 'lambda_f': 0.0001, 'lambda_s': 0.000005,
        'R0': 15.0, 'p_HIT': 0.93, 'AEFI_per_1000': 18.0,
        'booster_threshold': None, 'annual': False,
        'age_targets': ['0-5', '6-17'],
        'unit_cost_php': 320,
        'shelf_life_days': 730,
        'route': 'SC', 'site_of_admin': 'Upper arm',
    },
    'HPV': {
        'doses': 2, 'interval_days': 180,
        'E_peak': 0.95, 'phi': 0.15, 'lambda_f': 0.0002, 'lambda_s': 0.000008,
        'R0': 2.5, 'p_HIT': 0.60, 'AEFI_per_1000': 22.0,
        'booster_threshold': None, 'annual': False,
        'age_targets': ['6-17'],
        'unit_cost_php': 2800,
        'shelf_life_days': 1095,
        'route': 'IM', 'site_of_admin': 'Deltoid',
    },
    'Hepatitis_B': {
        'doses': 3, 'interval_days': 30,
        'E_peak': 0.90, 'phi': 0.20, 'lambda_f': 0.0003, 'lambda_s': 0.00001,
        'R0': 2.0, 'p_HIT': 0.50, 'AEFI_per_1000': 8.0,
        'booster_threshold': None, 'annual': False,
        'age_targets': ['0-5'],
        'unit_cost_php': 280,
        'shelf_life_days': 730,
        'route': 'IM', 'site_of_admin': 'Anterolateral thigh',
    },
    'Rabies_PEP': {
        'doses': 4, 'interval_days': 7,
        'E_peak': 0.99, 'phi': 0.30, 'lambda_f': 0.0008, 'lambda_s': 0.0002,
        'R0': None, 'p_HIT': None, 'AEFI_per_1000': 30.0,
        'booster_threshold': None, 'annual': False,
        'age_targets': ['0-5', '6-17', '18-59', '60+'],
        'unit_cost_php': 1200,
        'shelf_life_days': 365,
        'route': 'IM', 'site_of_admin': 'Deltoid',
    },
}

# ── Cold chain parameters ────────────────────────────────────────────────────
SITE_TYPES = ['Regional Hospital', 'RHU', 'Barangay Health Center', 'Mobile Outreach']
SITE_TYPE_WEIGHTS = [0.10, 0.35, 0.40, 0.15]
COLD_CHAIN_BREACH_PROB = {
    'Regional Hospital': 0.03, 'RHU': 0.08,
    'Barangay Health Center': 0.14, 'Mobile Outreach': 0.22,
}
COLD_CHAIN_TIER = {
    'Regional Hospital': 'Tier 1', 'RHU': 'Tier 2',
    'Barangay Health Center': 'Tier 3', 'Mobile Outreach': 'Mobile',
}
WASTAGE_PARAMS = {
    'None'  : (1.0,  19.0),
    'Minor' : (3.6,  16.4),
    'Major' : (4.5,   5.5),
}

# ── Philippine calendar event multipliers ────────────────────────────────────
PH_CALENDAR_EVENTS = [
    ((3, 28), (4,  3), 0.40, 'Holy Week'),
    ((6,  1), (7, 15), 1.60, 'Back-to-school immunization'),
    ((8,  1), (8, 31), 2.00, 'Sabayang Pagbabakuna'),
    ((11, 1), (11, 2), 0.30, 'Undas'),
    ((12,24), (1,  2), 0.25, 'Christmas-New Year'),
]

# ── AEFI profiles per vaccine ────────────────────────────────────────────────
AEFI_TYPES = {
    'COVID_Booster': ['Injection site pain','Fatigue','Headache','Myalgia','Chills','Fever'],
    'Influenza':     ['Injection site soreness','Low-grade fever','Myalgia','Headache'],
    'MMR':           ['Fever','Rash','Mild joint pain','Injection site pain','Lymphadenopathy'],
    'HPV':           ['Injection site pain','Dizziness','Nausea','Headache','Syncope'],
    'Hepatitis_B':   ['Injection site soreness','Fatigue','Low-grade fever'],
    'Rabies_PEP':    ['Injection site pain','Edema','Headache','Nausea','Fever'],
}

# ── SEC distribution (PSA 2022) ──────────────────────────────────────────────
SEC_CLASSES   = ['A', 'B', 'C', 'D', 'E']
SEC_WEIGHTS   = [0.01, 0.09, 0.22, 0.48, 0.20]
SEC_SCORE_MAP = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5}

# ── Age group distributions ──────────────────────────────────────────────────
AGE_GROUPS  = ['0-5', '6-17', '18-59', '60+']
AGE_WEIGHTS = [0.12,  0.23,   0.52,    0.13]

def setup_workspace():
    for sub in SUBFOLDERS:
        path = os.path.join(DATA_ROOT, sub)
        os.makedirs(path, exist_ok=True)
    print(f'✅ Workspace initialized at: {DATA_ROOT}/')
    for sub in SUBFOLDERS:
        print(f'   {DATA_ROOT}/{sub}/')

setup_workspace()
print(f'\n✅ Constants loaded. Simulation window: {START_DATE} → {END_DATE}')
print(f'   Beneficiaries : {N_PEOPLE:,} | Sites: {N_SITES} | Vaccines: {len(VACCINE_CONFIGS)}')
print(f'   Random seed   : {RANDOM_SEED} (change to generate natural variance)')

---
## 2. Helper Functions

**Description:** Shared utility functions used across all engine cells.

In [ ]:
def new_id(prefix):
    """Generate a short prefixed UUID."""
    return f'{prefix}-{str(uuid.uuid4())[:8].upper()}'

def random_date(start, end):
    """Uniform random date between start and end."""
    delta = (end - start).days
    return start + timedelta(days=int(np.random.randint(0, delta)))

def get_season(d):
    """Return Philippine season for a given date (PAGASA classification)."""
    m = d.month
    if m in [11, 12, 1, 2]:   return 'AMIHAN'
    elif m in [3, 4, 5]:       return 'SUMMER'
    else:                      return 'HABAGAT'

def calendar_multiplier(d):
    """Return activity multiplier for Philippine health calendar events."""
    for (sm, sd), (em, ed), mult, _ in PH_CALENDAR_EVENTS:
        start = date(d.year, sm, sd)
        end_y = d.year if em >= sm else d.year + 1
        end   = date(end_y, em, ed)
        if start <= d <= end:
            return mult
    return 1.0

def seasonal_volume(d, base, psi=0.20, t_peak_month=8):
    """Engine H: seasonal appointment volume with sine forcing."""
    t = (d - date(d.year, 1, 1)).days
    t_peak = (date(d.year, t_peak_month, 1) - date(d.year, 1, 1)).days
    sine_factor = 1 + psi * math.sin(2 * math.pi * (t - t_peak) / 365)
    return base * sine_factor * calendar_multiplier(d)

def access_barrier(distance_km, sec_score, rurality):
    """Composite structural-access barrier ∈ [0,1].
    Drives the HETEROGENEOUS reminder effect: high-barrier patients (rural, low-SEC,
    far from site) benefit most from a reminder."""
    return float(np.clip(
        0.40 * rurality
        + 0.30 * (sec_score / 5)
        + 0.30 * min(distance_km / 20, 1.0),
        0.0, 1.0
    ))

def logistic_noshow(distance_km, sec_score, dose_num, prior_noshow, rurality,
                    reminder=0, barrier=0.0):
    """Engine B: logistic model for appointment no-show probability.
    `reminder` is the causal treatment. Its effect is heterogeneous in `barrier`:
        Δlog-odds = −(REMINDER_BASE + REMINDER_HETERO · barrier)
    so the conditional average treatment effect (CATE) varies across subgroups."""
    logit = (
        BETA['intercept']
        + BETA['distance']     * distance_km
        + BETA['sec']          * sec_score
        + BETA['dose_num']     * (dose_num - 1)
        + BETA['prior_noshow'] * int(prior_noshow)
        + BETA['rurality']     * rurality
    )
    if reminder:
        logit -= (REMINDER_BASE + REMINDER_HETERO * barrier)
    return 1 / (1 + math.exp(-logit))

def bi_exponential_efficacy(t, E_peak, phi, lambda_f, lambda_s, dose_num=1):
    """Engine C: bi-exponential waning efficacy model."""
    dose_prime = 1 + 0.05 * (dose_num - 1)
    E = E_peak * dose_prime * (phi * math.exp(-lambda_f * t) + (1 - phi) * math.exp(-lambda_s * t))
    return min(E, 1.0)

def arrhenius_potency(excursion_temp_c, duration_hr, Ea=60000, A=1e10):
    """Engine D: Arrhenius vaccine potency retention after cold chain breach."""
    R = 8.314
    T_kelvin = excursion_temp_c + 273.15
    k = A * math.exp(-Ea / (R * T_kelvin))
    potency = math.exp(-k * duration_hr * 3600)
    return max(round(potency, 4), 0.0)

def vvm_status(potency):
    """WHO Vaccine Vial Monitor stage from potency retention."""
    if potency >= 0.95: return 'Stage 1'
    elif potency >= 0.80: return 'Stage 2'
    elif potency >= 0.50: return 'Stage 3'
    else: return 'Stage 4'

def risk_tier(potency, breach):
    """Cold chain risk classification."""
    if not breach: return 'Low'
    if potency >= 0.85: return 'Medium'
    elif potency >= 0.60: return 'High'
    return 'Critical'

print('✅ Helper functions loaded.')
print('   logistic_noshow(), bi_exponential_efficacy(), arrhenius_potency()')
print('   seasonal_volume(), calendar_multiplier(), get_season()')
print('   vvm_status(), risk_tier(), new_id(), random_date()')

---
## 3. Engine A — Sites Generator

**Focus:** Generates `dim_sites` — 150 vaccination facilities across all 17 PH regions,  
with realistic capacity distributions, cold chain tiers, and rurality indices.

In [ ]:
# ── Approximate region centroids (lat, lon) ───────────────────────────────────
REGION_COORDS = {
    'NCR': (14.599, 120.984), 'CAR': (17.350, 121.170), 'Region I': (16.080, 120.620),
    'Region II': (16.970, 121.810), 'Region III': (15.480, 120.710), 'Region IV-A': (14.100, 121.100),
    'MIMAROPA': (10.640, 119.980), 'Region V': (13.420, 123.410), 'Region VI': (11.000, 122.570),
    'Region VII': (10.300, 123.890), 'Region VIII': (11.250, 124.990), 'Region IX': (7.830, 123.290),
    'Region X': (8.020, 124.690), 'Region XI': (7.190, 125.460), 'Region XII': (6.700, 124.850),
    'CARAGA': (8.950, 125.540), 'BARMM': (7.200, 124.240),
}

SITE_TYPE_OPS_DAYS = {
    'Regional Hospital': 7, 'RHU': 5,
    'Barangay Health Center': 5, 'Mobile Outreach': 3,
}
SITE_TYPE_CAPACITY_PARAMS = {
    'Regional Hospital': (150, 30), 'RHU': (80, 20),
    'Barangay Health Center': (40, 12), 'Mobile Outreach': (25, 8),
}

def generate_dim_sites(n=N_SITES):
    rows = []
    for i in range(n):
        region = np.random.choice(REGIONS, p=REGION_WEIGHTS)
        province = np.random.choice(REGION_PROVINCES[region])
        site_type = np.random.choice(SITE_TYPES, p=SITE_TYPE_WEIGHTS)
        lat_c, lon_c = REGION_COORDS[region]
        rurality = REGION_RURALITY[region] + np.random.uniform(-0.08, 0.08)
        rurality = float(np.clip(rurality, 0.0, 1.0))
        mu, sigma = SITE_TYPE_CAPACITY_PARAMS[site_type]
        capacity = int(np.clip(np.random.normal(mu, sigma), 10, 300))
        rows.append({
            'site_id'            : new_id('PHV'),
            'site_name'          : f'{province} {site_type} {i+1}',
            'region'             : region,
            'province'           : province,
            'city_municipality'  : fake.city(),
            'latitude'           : round(lat_c + np.random.uniform(-0.5, 0.5), 4),
            'longitude'          : round(lon_c + np.random.uniform(-0.5, 0.5), 4),
            'site_type'          : site_type,
            'cold_chain_capacity': capacity,
            'operational_days_pw': SITE_TYPE_OPS_DAYS[site_type],
            'rurality_index'     : round(rurality, 3),
            'doh_accredited'     : bool(np.random.binomial(1, 0.92)),
            'cold_chain_tier'    : COLD_CHAIN_TIER[site_type],
            'established_year'   : int(np.random.randint(1980, 2023)),
        })
    return pd.DataFrame(rows)

dim_sites = generate_dim_sites()
print(f'✅ dim_sites generated: {len(dim_sites):,} rows × {len(dim_sites.columns)} columns')
print(f'\nSite type distribution:')
print(dim_sites['site_type'].value_counts().to_string())
print(f'\nRegion coverage: {dim_sites["region"].nunique()} / 17 regions')
dim_sites.head(3)

---
## 4. Engine B — People Generator

**Focus:** Generates `dim_people` — 5,000 beneficiaries distributed across regions  
using Poisson-weighted PSA 2024 population estimates. Assigns each person  
to their nearest site and computes their logistic distance to it.

In [ ]:
COMORBIDITY_BY_AGE = {'0-5': 0.04, '6-17': 0.06, '18-59': 0.18, '60+': 0.42}
PHILHEALTH_BY_SEC  = {'A': 0.99, 'B': 0.95, 'C': 0.80, 'D': 0.55, 'E': 0.28}

def assign_vaccines(age_group, sex):
    """Return list of vaccines this person is eligible for."""
    vx = []
    for vname, vcfg in VACCINE_CONFIGS.items():
        if age_group in vcfg['age_targets']:
            if vname == 'HPV' and sex == 'Male':
                continue  # HPV campaign targets females in PH EPI
            vx.append(vname)
    return vx

def age_from_group(age_group):
    ranges = {'0-5': (0,5), '6-17': (6,17), '18-59': (18,59), '60+': (60,90)}
    lo, hi = ranges[age_group]
    return int(np.random.randint(lo, hi+1))

def generate_dim_people(n=N_PEOPLE, sites_df=None):
    rows = []
    for _ in range(n):
        region    = np.random.choice(REGIONS, p=REGION_WEIGHTS)
        province  = np.random.choice(REGION_PROVINCES[region])
        sec       = np.random.choice(SEC_CLASSES, p=SEC_WEIGHTS)
        age_group = np.random.choice(AGE_GROUPS, p=AGE_WEIGHTS)
        sex       = np.random.choice(['Male','Female'], p=[0.492, 0.508])
        age       = age_from_group(age_group)
        dob       = date(2024,1,1) - timedelta(days=age*365 + np.random.randint(0,364))
        rurality  = REGION_RURALITY[region]
        # Distance: log-normal scaled by rurality
        dist_km   = round(float(np.random.lognormal(mean=1.2, sigma=0.8) * (0.5 + rurality)), 2)
        # Assign to a site in same region (or closest)
        region_sites = sites_df[sites_df['region'] == region]
        if len(region_sites) == 0:
            region_sites = sites_df
        site_row  = region_sites.sample(1, weights='cold_chain_capacity').iloc[0]
        prior_hx  = np.random.choice(['None','Partial','Complete'], p=[0.20, 0.45, 0.35])
        rows.append({
            'person_id'           : new_id('PH'),
            'full_name'           : fake.name(),
            'date_of_birth'       : dob.isoformat(),
            'age_group'           : age_group,
            'age_at_registration' : age,
            'sex'                 : sex,
            'region'              : region,
            'province'            : province,
            'city_municipality'   : fake.city(),
            'barangay'            : fake.street_name(),
            'socioeconomic_class' : sec,
            'priority_group'      : (
                'Healthcare Worker' if np.random.random() < 0.08
                else 'Elderly' if age_group == '60+'
                else 'Indigent' if sec == 'E'
                else 'General Population'
            ),
            'philhealth_member'   : bool(np.random.binomial(1, PHILHEALTH_BY_SEC[sec])),
            'distance_to_site_km' : dist_km,
            'assigned_site_id'    : site_row['site_id'],
            'registration_date'   : random_date(date(2024,1,1), date(2024,6,30)).isoformat(),
            'has_comorbidity'     : bool(np.random.binomial(1, COMORBIDITY_BY_AGE[age_group])),
            'prior_vaccination_hx': prior_hx,
            'eligible_vaccines'   : '|'.join(assign_vaccines(age_group, sex)),
            'rurality_index'      : round(rurality, 3),
        })
    return pd.DataFrame(rows)

dim_people = generate_dim_people(N_PEOPLE, dim_sites)
print(f'✅ dim_people generated: {len(dim_people):,} rows × {len(dim_people.columns)} columns')
print(f'\nAge group distribution:')
print(dim_people['age_group'].value_counts().to_string())
print(f'\nSEC distribution:')
print(dim_people['socioeconomic_class'].value_counts().sort_index().to_string())
dim_people.head(3)

---
## 5. Engine C — Appointments Generator

**Focus:** Generates `fact_appointments` using the logistic no-show model (Engine B)
and seasonal forcing (Engine H). Each person receives appointments for all
eligible vaccines across the 5-year window, with multi-dose sequencing.

**Causal design (`reminder_sent`).** The reminder is the dataset's built-in
*treatment variable*. It is drawn at random — `Bernoulli(0.65)`, independent of every
covariate — **before** the no-show outcome is sampled. This satisfies the ignorability
assumption $Y(0),Y(1)\perp T\mid X$, so the average and conditional treatment effects
(ATE / CATE, analyzed in notebook `02`) are *identified* without an instrument.

The treatment effect is **heterogeneous**: a reminder lowers no-show log-odds by
$(\text{REMINDER\_BASE} + \text{REMINDER\_HETERO}\cdot \text{barrier})$, where
`barrier_score` ∈ [0,1] captures structural access difficulty (rurality, low SEC,
distance). High-barrier patients benefit most — producing a non-trivial CATE that the
optimal-policy analysis exploits. The ground-truth ATE embedded here is recoverable
by IPW / doubly-robust estimation downstream (≈ +0.12 reduction in no-show probability).

In [ ]:
def generate_fact_appointments(people_df, sites_df):
    rows = []
    site_map = sites_df.set_index('site_id')[['rurality_index','region']].to_dict('index')

    for _, person in people_df.iterrows():
        eligible = person['eligible_vaccines'].split('|') if person['eligible_vaccines'] else []
        sec_score   = SEC_SCORE_MAP[person['socioeconomic_class']]
        dist_km     = person['distance_to_site_km']
        site_id     = person['assigned_site_id']
        rurality    = site_map.get(site_id, {}).get('rurality_index', 0.5)
        prior_noshow = person['prior_vaccination_hx'] == 'None'
        barrier     = access_barrier(dist_km, sec_score, rurality)  # heterogeneous-effect driver

        for vaccine in eligible:
            cfg = VACCINE_CONFIGS[vaccine]
            n_doses     = cfg['doses']
            interval    = cfg['interval_days'] or 30
            is_annual   = cfg['annual']
            years_range = range(2024, 2029) if is_annual else [2024]

            for yr in years_range:
                # First dose: random date in campaign year, modulated by seasonal volume
                base_date = random_date(
                    max(date(yr, 1, 1), START_DATE),
                    min(date(yr, 12, 1), END_DATE)
                )
                prior_ns = prior_noshow
                for dose_n in range(1, n_doses + 1):
                    sched_date = base_date + timedelta(days=interval*(dose_n-1))
                    if sched_date > END_DATE:
                        break
                    # Treatment assigned at random (as-if RCT) BEFORE the outcome,
                    # so it causally affects no-show and ATE/CATE are identified.
                    reminder = int(np.random.binomial(1, 0.65))
                    p_ns   = logistic_noshow(dist_km, sec_score, dose_n, prior_ns,
                                             rurality, reminder, barrier)
                    # Seasonality modulates appointment VOLUME (Engine H), NOT a
                    # person's no-show odds. p_ns is the pure logistic output.
                    p_ns_adj = float(np.clip(p_ns, 0.02, 0.90))
                    noshow   = bool(np.random.binomial(1, p_ns_adj))
                    if noshow:
                        reschedule = bool(np.random.binomial(1, 0.35))
                        status = 'Rescheduled' if reschedule else 'No-show'
                    else:
                        status = 'Kept'
                    rows.append({
                        'appt_id'           : new_id('APT'),
                        'person_id'         : person['person_id'],
                        'site_id'           : site_id,
                        'vaccine_type'      : vaccine,
                        'dose_number'       : dose_n,
                        'scheduled_date'    : sched_date.isoformat(),
                        'scheduled_year'    : sched_date.year,
                        'scheduled_month'   : sched_date.month,
                        'season'            : get_season(sched_date),
                        'status'            : status,
                        'noshow_probability': round(p_ns_adj, 4),
                        'is_catch_up'       : dose_n > 1 and prior_ns,
                        'campaign_phase'    : (
                            'Phase 1' if sched_date.year <= 2024
                            else 'Phase 2' if sched_date.year <= 2025
                            else 'Phase 3' if sched_date.year <= 2026
                            else 'Maintenance'
                        ),
                        'appointment_source': np.random.choice(
                            ['Pre-registered','Walk-in','Community outreach','Referral'],
                            p=[0.45, 0.30, 0.15, 0.10]
                        ),
                        'reminder_sent'     : bool(reminder),    # ← causal treatment
                        'barrier_score'     : round(barrier, 4), # ← drives heterogeneous CATE
                        'distance_km'       : dist_km,
                    })
                    prior_ns = noshow  # Prior no-show carries forward
    return pd.DataFrame(rows)

print('Generating fact_appointments (this may take ~30s)...')
fact_appointments = generate_fact_appointments(dim_people, dim_sites)
print(f'✅ fact_appointments: {len(fact_appointments):,} rows × {len(fact_appointments.columns)} cols')
print(f'\nAppointment status distribution:')
print(fact_appointments['status'].value_counts().to_string())
overall_noshow = (fact_appointments['status'] != 'Kept').mean()
print(f'\nOverall no-show rate: {overall_noshow:.1%} (target: ~16%)')
fact_appointments.head(3)

---
## 6. Engine D — Inventory & Cold Chain Generator

**Focus:** Generates `fact_inventory_shipments` using Arrhenius potency model,  
Beta-distributed wastage, and discrete-time stock balance (Engine G).

In [ ]:
def generate_inventory_shipments(sites_df, appts_df):
    rows = []
    # Track running stock per site × vaccine
    stock = {}

    # Generate ~8 shipments per site per vaccine per year
    for _, site in sites_df.iterrows():
        site_id   = site['site_id']
        region    = site['region']
        site_type = site['site_type']
        breach_p  = COLD_CHAIN_BREACH_PROB[site_type]
        source_hub= f'DOH-{region[:6]}-Hub'

        for vaccine, vcfg in VACCINE_CONFIGS.items():
            stock_key = (site_id, vaccine)
            stock[stock_key] = int(np.random.randint(50, 200))

            # ~8 shipments/year × 5 years
            for yr in range(2024, 2029):
                n_shipments = int(np.random.poisson(8))
                for sh in range(n_shipments):
                    ship_date = random_date(date(yr,1,1), date(yr,12,20))
                    transit   = int(np.random.negative_binomial(2, 0.4)) + 1
                    recv_date = min(ship_date + timedelta(days=transit), date(yr,12,31))
                    ordered   = int(np.random.negative_binomial(5, 0.3)) + 50
                    received  = int(ordered * np.random.uniform(0.80, 1.05))
                    mfr_delta = int(np.random.randint(30, min(180, vcfg['shelf_life_days']//2)))
                    mfr_date  = ship_date - timedelta(days=mfr_delta)
                    exp_date  = mfr_date + timedelta(days=vcfg['shelf_life_days'])

                    # Cold chain breach
                    breach    = bool(np.random.binomial(1, breach_p))
                    if breach:
                        breach_type  = np.random.choice(['Minor','Major'], p=[0.70,0.30])
                        exc_temp     = round(float(np.random.normal(10 if breach_type=='Minor' else 20, 3)), 1)
                        exc_dur      = round(float(np.random.exponential(scale=6 if breach_type=='Minor' else 12)), 1)
                        potency      = arrhenius_potency(exc_temp, exc_dur)
                        w_alpha, w_beta = WASTAGE_PARAMS[breach_type]
                    else:
                        breach_type  = 'None'
                        exc_temp     = None
                        exc_dur      = None
                        potency      = round(float(np.random.uniform(0.95, 1.0)), 4)
                        w_alpha, w_beta = WASTAGE_PARAMS['None']

                    wastage_rate   = round(float(np.random.beta(w_alpha, w_beta)), 4)
                    wasted         = int(received * wastage_rate)
                    available      = received - wasted
                    prev_stock     = stock[stock_key]
                    stock[stock_key] = prev_stock + available
                    safety_stock   = max(10, int(site['cold_chain_capacity'] * 0.10))

                    rows.append({
                        'shipment_id'         : new_id('SHP'),
                        'site_id'             : site_id,
                        'vaccine_type'        : vaccine,
                        'source_hub'          : source_hub,
                        'shipment_date'       : ship_date.isoformat(),
                        'received_date'       : recv_date.isoformat(),
                        'transit_days'        : transit,
                        'doses_ordered'       : ordered,
                        'doses_received'      : received,
                        'fulfillment_rate'    : round(received / ordered, 4),
                        'manufacture_date'    : mfr_date.isoformat(),
                        'expiry_date'         : exp_date.isoformat(),
                        'cold_chain_breach'   : breach,
                        'breach_type'         : breach_type,
                        'excursion_temp_c'    : exc_temp,
                        'excursion_duration_hr': exc_dur,
                        'potency_retained'    : potency,
                        'wastage_rate'        : wastage_rate,
                        'doses_wasted'        : wasted,
                        'doses_available'     : available,
                        'stock_on_hand_before': prev_stock,
                        'stock_on_hand_after' : stock[stock_key],
                        'stockout_flag'       : stock[stock_key] < safety_stock,
                        'vvm_status'          : vvm_status(potency),
                    })
    return pd.DataFrame(rows)

print('Generating fact_inventory_shipments...')
fact_inventory = generate_inventory_shipments(dim_sites, fact_appointments)
print(f'✅ fact_inventory_shipments: {len(fact_inventory):,} rows × {len(fact_inventory.columns)} cols')
breach_rate = fact_inventory['cold_chain_breach'].mean()
avg_wastage = fact_inventory['wastage_rate'].mean()
print(f'\nCold chain breach rate : {breach_rate:.1%}')
print(f'Average wastage rate   : {avg_wastage:.1%}')
print(f'VVM status breakdown:')
print(fact_inventory['vvm_status'].value_counts().to_string())
fact_inventory.head(3)

---
## 7. Engine E — Vaccination Events Generator

**Focus:** Generates `fact_vaccination_events` from kept appointments only.  
Applies bi-exponential efficacy (Engine C), Arrhenius potency from linked  
shipments (Engine D), and age-stratified AEFI model (Engine E).

In [ ]:
AEFI_SEVERITY_PROBS = {'mild': 0.88, 'moderate': 0.10, 'severe': 0.02}
STAFF_TYPES = ['Nurse','Doctor','Midwife','Health Aide']
STAFF_WEIGHTS = [0.55, 0.20, 0.18, 0.07]

def generate_vaccination_events(appts_df, inventory_df, people_df):
    kept = appts_df[appts_df['status'] == 'Kept'].copy()

    # Build shipment lookup: site × vaccine → list of shipments (sorted by date)
    inv_lookup = {}
    for _, sh in inventory_df.iterrows():
        key = (sh['site_id'], sh['vaccine_type'])
        inv_lookup.setdefault(key, []).append(sh)

    # Build person lookup
    people_map = people_df.set_index('person_id').to_dict('index')

    rows = []
    for _, appt in kept.iterrows():
        vaccine   = appt['vaccine_type']
        cfg       = VACCINE_CONFIGS[vaccine]
        dose_n    = int(appt['dose_number'])
        person    = people_map.get(appt['person_id'], {})
        age_group = person.get('age_group', '18-59')

        # Admin date: scheduled ± small jitter (1–2 days)
        sched_dt  = date.fromisoformat(appt['scheduled_date'])
        jitter    = int(np.random.normal(0, 1.5))
        admin_dt  = sched_dt + timedelta(days=jitter)
        admin_dt  = max(min(admin_dt, END_DATE), START_DATE)

        # Link to a shipment for this site × vaccine
        key       = (appt['site_id'], vaccine)
        shipments = inv_lookup.get(key, [])
        shipment  = shipments[np.random.randint(0, max(len(shipments),1))] if shipments else None
        potency   = float(shipment['potency_retained']) if shipment is not None else 0.95
        sh_id     = shipment['shipment_id'] if shipment is not None else None
        cold_ok   = not bool(shipment['cold_chain_breach']) if shipment is not None else True
        exp_date  = shipment['expiry_date'] if shipment is not None else (admin_dt + timedelta(days=180)).isoformat()
        lot_num   = f"{vaccine[:3].upper()}-{admin_dt.year}-B{np.random.randint(1,99):02d}"

        # Efficacy: bi-exponential × potency × dose priming
        eff_at_admin = bi_exponential_efficacy(
            t=0, E_peak=cfg['E_peak'] * potency,
            phi=cfg['phi'], lambda_f=cfg['lambda_f'],
            lambda_s=cfg['lambda_s'], dose_num=dose_n
        )

        # Series completion
        series_complete = dose_n >= cfg['doses']
        next_due = None
        if not series_complete and cfg['interval_days']:
            nd = admin_dt + timedelta(days=cfg['interval_days'])
            if nd <= END_DATE:
                next_due = nd.isoformat()

        # AEFI: Poisson rate → severity → type
        aefi_rate   = cfg['AEFI_per_1000'] / 1000
        aefi_count  = np.random.poisson(aefi_rate)
        aefi_rep    = aefi_count > 0
        if aefi_rep:
            sev_draw = np.random.random()
            if sev_draw < AEFI_SEVERITY_PROBS['severe']:
                sev = 'Severe'
            elif sev_draw < AEFI_SEVERITY_PROBS['severe'] + AEFI_SEVERITY_PROBS['moderate']:
                sev = 'Moderate'
            else:
                sev = 'Mild'
            aefi_type = np.random.choice(AEFI_TYPES.get(vaccine, ['Injection site pain']))
            resolved  = bool(np.random.binomial(1, 0.97 if sev=='Mild' else 0.85 if sev=='Moderate' else 0.60))
        else:
            sev = None; aefi_type = None; resolved = None

        rows.append({
            'event_id'            : new_id('EVT'),
            'appt_id'             : appt['appt_id'],
            'person_id'           : appt['person_id'],
            'site_id'             : appt['site_id'],
            'vaccine_type'        : vaccine,
            'dose_number'         : dose_n,
            'administered_date'   : admin_dt.isoformat(),
            'lot_number'          : lot_num,
            'expiry_date'         : exp_date,
            'administered_by'     : np.random.choice(STAFF_TYPES, p=STAFF_WEIGHTS),
            'route'               : cfg['route'],
            'anatomical_site'     : cfg['site_of_admin'],
            'potency_at_admin'    : round(potency, 4),
            'efficacy_at_admin'   : round(eff_at_admin, 4),
            'days_since_prior_dose': None if dose_n == 1 else cfg['interval_days'],
            'series_complete'     : series_complete,
            'next_dose_due_date'  : next_due,
            'aefi_reported'       : aefi_rep,
            'aefi_severity'       : sev,
            'aefi_type'           : aefi_type,
            'aefi_resolved'       : resolved,
            'cold_chain_ok'       : cold_ok,
            'linked_shipment_id'  : sh_id,
        })
    return pd.DataFrame(rows)

print('Generating fact_vaccination_events...')
fact_events = generate_vaccination_events(fact_appointments, fact_inventory, dim_people)
print(f'✅ fact_vaccination_events: {len(fact_events):,} rows × {len(fact_events.columns)} cols')
print(f'\nSeries completion rate  : {fact_events["series_complete"].mean():.1%}')
print(f'AEFI reported rate      : {fact_events["aefi_reported"].mean():.1%}')
print(f'Cold chain ok rate      : {fact_events["cold_chain_ok"].mean():.1%}')
print(f'\nAEFI severity breakdown:')
print(fact_events['aefi_severity'].value_counts(dropna=False).to_string())
fact_events.head(3)

---
## 8. Engine F — Gold Layer Generator

**Focus:** Derives three analytical gold tables from the silver layer:  
`gold_coverage`, `gold_efficacy_waning`, and `gold_cold_chain_risk`.

In [ ]:
def generate_gold_coverage(events_df, appts_df, people_df, sites_df):
    """Regional coverage rates per vaccine per quarter, with herd immunity metrics."""
    rows = []
    site_region = sites_df.set_index('site_id')['region'].to_dict()
    events_df   = events_df.copy()
    events_df['region'] = events_df['site_id'].map(site_region)
    events_df['year']   = pd.to_datetime(events_df['administered_date']).dt.year
    events_df['quarter']= pd.to_datetime(events_df['administered_date']).dt.quarter
    region_people_count = people_df['region'].value_counts().to_dict()

    for region in REGIONS:
        reg_pop = region_people_count.get(region, 1)
        for vaccine, vcfg in VACCINE_CONFIGS.items():
            age_targets = vcfg['age_targets']
            target_pop  = int(people_df[
                (people_df['region'] == region) &
                (people_df['age_group'].isin(age_targets))
            ].shape[0])
            if target_pop == 0: target_pop = 1

            for yr in range(2024, 2029):
                for qtr in range(1, 5):
                    subset = events_df[
                        (events_df['region'] == region) &
                        (events_df['vaccine_type'] == vaccine) &
                        (events_df['year'] == yr) &
                        (events_df['quarter'] == qtr)
                    ]
                    doses_given   = len(subset)
                    series_done   = subset['series_complete'].sum()
                    coverage_rate = round(series_done / target_pop, 4)
                    # Effective coverage: mean efficacy at time of admin × coverage
                    eff_cov   = round(float(subset['efficacy_at_admin'].mean() * coverage_rate)
                                      if len(subset) > 0 else 0.0, 4)
                    p_HIT     = vcfg.get('p_HIT') or 0.0
                    R0        = vcfg.get('R0') or 1.0
                    R_eff     = round(R0 * (1 - eff_cov), 4)
                    hi_gap    = round(p_HIT - eff_cov, 4)
                    ns_count  = appts_df[
                        (appts_df['site_id'].isin(sites_df[sites_df['region']==region]['site_id'])) &
                        (appts_df['vaccine_type'] == vaccine) &
                        (appts_df['scheduled_year'] == yr) &
                        (appts_df['scheduled_month'].between((qtr-1)*3+1, qtr*3))
                    ]
                    no_show_rate = round((ns_count['status'] != 'Kept').mean()
                                         if len(ns_count) > 0 else 0.0, 4)
                    inv_sub = fact_inventory[
                        (fact_inventory['site_id'].isin(sites_df[sites_df['region']==region]['site_id'])) &
                        (fact_inventory['vaccine_type'] == vaccine)
                    ]
                    rows.append({
                        'coverage_id'       : new_id('COV'),
                        'region'            : region,
                        'vaccine_type'      : vaccine,
                        'period_year'       : yr,
                        'period_quarter'    : qtr,
                        'target_population' : target_pop,
                        'doses_administered': doses_given,
                        'series_completed'  : int(series_done),
                        'coverage_rate'     : coverage_rate,
                        'effective_coverage': eff_cov,
                        'p_HIT'             : p_HIT,
                        'herd_immunity_gap' : hi_gap,
                        'R_eff'             : R_eff,
                        'no_show_rate'      : no_show_rate,
                        'wastage_rate_avg'  : round(float(inv_sub['wastage_rate'].mean()), 4)
                                             if len(inv_sub) > 0 else 0.0,
                    })
    return pd.DataFrame(rows)

def generate_gold_efficacy_waning(events_df, people_df):
    """Monthly efficacy snapshots per vaccinated person — Engine C waning model.
    VECTORIZED with NumPy broadcasting: computes all people x 60 months in one
    matrix op instead of a nested Python loop (~30x faster, O(N) memory)."""
    completed = events_df[events_df['series_complete'] == True].copy()
    if len(completed) == 0:
        return pd.DataFrame()
    end_ord = END_DATE.toordinal()
    months  = np.arange(1, 61)
    t_row   = (months * 30).astype(float)              # (60,) days since final dose
    frames  = []
    # Group by vaccine so decay constants are shared across the broadcast
    for vaccine, grp in completed.groupby('vaccine_type'):
        cfg       = VACCINE_CONFIGS[vaccine]
        final_ord = grp['administered_date'].map(lambda x: date.fromisoformat(x).toordinal()).to_numpy()
        dose_n    = grp['dose_number'].astype(int).to_numpy()
        pid       = grp['person_id'].to_numpy()
        snap_ord  = final_ord[:, None] + t_row[None, :] # (N, 60) snapshot ordinals
        valid     = snap_ord <= end_ord                 # within 2028 horizon
        dprime    = (1 + 0.05 * (dose_n - 1))[:, None]  # (N, 1) dose-priming factor
        # Bi-exponential decay evaluated for every (person, month) at once
        E = cfg['E_peak'] * dprime * (
            cfg['phi'] * np.exp(-cfg['lambda_f'] * t_row[None, :]) +
            (1 - cfg['phi']) * np.exp(-cfg['lambda_s'] * t_row[None, :])
        )
        E = np.clip(E, 0.0, 1.0)
        ri, cj    = np.where(valid)                     # flatten valid cells
        eff       = E[ri, cj]
        t_flat    = t_row[cj].astype(int)
        snap_flat = snap_ord[ri, cj].astype(int)
        bt        = cfg.get('booster_threshold')
        frames.append(pd.DataFrame({
            'waning_id'            : [new_id('WAN') for _ in range(ri.size)],
            'person_id'            : pid[ri],
            'vaccine_type'         : vaccine,
            'final_dose_date'      : [date.fromordinal(int(o)).isoformat() for o in final_ord[ri]],
            'snapshot_date'        : [date.fromordinal(int(o)).isoformat() for o in snap_flat],
            'days_since_final_dose': t_flat,
            'estimated_protection' : np.round(eff, 4),
            'booster_recommended'  : (eff < bt) if bt is not None else np.zeros(ri.size, bool),
            'protection_tier'      : np.where(eff >= 0.80, 'High',
                                      np.where(eff >= 0.50, 'Moderate', 'Low')),
        }))
    return pd.concat(frames, ignore_index=True)


def generate_gold_cold_chain_risk(inventory_df, sites_df):
    """Risk classification per shipment."""
    rows = []
    for _, sh in inventory_df.iterrows():
        potency = float(sh['potency_retained'])
        breach  = bool(sh['cold_chain_breach'])
        vax     = sh['vaccine_type']
        cost    = VACCINE_CONFIGS[vax]['unit_cost_php']
        affected= int(sh['doses_received'] * (1 - potency))
        tier    = risk_tier(potency, breach)
        action  = (
            'None' if tier == 'Low'
            else 'Monitor' if tier == 'Medium'
            else 'Quarantine' if tier == 'High'
            else 'Discard'
        )
        rows.append({
            'risk_id'          : new_id('CCR'),
            'site_id'          : sh['site_id'],
            'shipment_id'      : sh['shipment_id'],
            'vaccine_type'     : vax,
            'potency_retained' : potency,
            'doses_affected'   : affected,
            'financial_loss_php': round(sh['doses_wasted'] * cost, 2),
            'risk_tier'        : tier,
            'action_required'  : action,
        })
    return pd.DataFrame(rows)

print('Generating gold layer tables...')
gold_coverage = generate_gold_coverage(fact_events, fact_appointments, dim_people, dim_sites)
print(f'✅ gold_coverage          : {len(gold_coverage):,} rows')
print('Generating gold_efficacy_waning (largest table — may take ~60s)...')
gold_waning   = generate_gold_efficacy_waning(fact_events, dim_people)
print(f'✅ gold_efficacy_waning   : {len(gold_waning):,} rows')
gold_cc_risk  = generate_gold_cold_chain_risk(fact_inventory, dim_sites)
print(f'✅ gold_cold_chain_risk   : {len(gold_cc_risk):,} rows')
print(f'\nNational avg coverage rate (all vaccines/years):')
print(gold_coverage.groupby('vaccine_type')['coverage_rate'].mean().round(3).to_string())

---
## 9. Master Factory

**Focus:** Orchestration cell that saves all tables as CSVs to the structured  
directory and prints a final summary report of the complete dataset.

In [ ]:
import time
t0 = time.time()

# ── Save silver layer ─────────────────────────────────────────────────────────
silver_tables = {
    'dim_sites'                : dim_sites,
    'dim_people'               : dim_people,
    'fact_appointments'        : fact_appointments,
    'fact_vaccination_events'  : fact_events,
    'fact_inventory_shipments' : fact_inventory,
}
for name, df in silver_tables.items():
    path = os.path.join(DATA_ROOT, 'observable', f'{name}.csv')
    df.to_csv(path, index=False)
    print(f'  ✅ Saved: {path:<55} {len(df):>8,} rows')

# ── Save gold layer ──────────────────────────────────────────────────────────
gold_tables = {
    'gold_coverage'          : gold_coverage,
    'gold_efficacy_waning'   : gold_waning,
    'gold_cold_chain_risk'   : gold_cc_risk,
}
for name, df in gold_tables.items():
    path = os.path.join(DATA_ROOT, 'gold', f'{name}.csv')
    df.to_csv(path, index=False)
    print(f'  ✅ Saved: {path:<55} {len(df):>8,} rows')

t1 = time.time()

# ── Final summary ─────────────────────────────────────────────────────────────
total_rows = sum(len(df) for df in {**silver_tables, **gold_tables}.values())
total_cols = sum(len(df.columns) for df in {**silver_tables, **gold_tables}.values())

print(f'\n{"="*60}')
print(f'  DATASET GENERATION COMPLETE')
print(f'  Random seed    : {RANDOM_SEED}')
print(f'  Campaign window: {START_DATE} → {END_DATE}')
print(f'  Total tables   : {len(silver_tables) + len(gold_tables)}')
print(f'  Total rows     : {total_rows:,}')
print(f'  Total columns  : {total_cols}')
print(f'  Generation time: {t1-t0:.1f}s')
print(f'{"="*60}')
print(f'\nKey metrics summary:')
print(f'  Beneficiaries registered    : {len(dim_sites):,} sites | {len(dim_people):,} people')
print(f'  Total appointments          : {len(fact_appointments):,}')
print(f'  Overall no-show rate        : {(fact_appointments["status"]!="Kept").mean():.1%}')
print(f'  Vaccinations administered   : {len(fact_events):,}')
print(f'  Series completion rate      : {fact_events["series_complete"].mean():.1%}')
print(f'  AEFI reported rate          : {fact_events["aefi_reported"].mean():.1%}')
print(f'  Cold chain breach rate      : {fact_inventory["cold_chain_breach"].mean():.1%}')
print(f'  Avg vaccine wastage rate    : {fact_inventory["wastage_rate"].mean():.1%}')
print(f'  Waning snapshots generated  : {len(gold_waning):,}')
print(f'\n  To regenerate with variance: change RANDOM_SEED in Cell 1 and Run All')

---
## 10. Inference Efficiency Benchmark

**Focus:** Profiles generation throughput and memory so the dataset can be
validated as a production fail-safe (fast to regenerate, train, and test).
Reports wall-time, peak memory, and rows/second.

In [ ]:
import time, tracemalloc

# Re-time the full pipeline with memory profiling
tracemalloc.start()
_t0 = time.perf_counter()

_sites = generate_dim_sites()
_people = generate_dim_people(N_PEOPLE, _sites)
_appts = generate_fact_appointments(_people, _sites)
_inv   = generate_inventory_shipments(_sites, _appts)
_events= generate_vaccination_events(_appts, _inv, _people)
_t_silver = time.perf_counter()

_cov = generate_gold_coverage(_events, _appts, _people, _sites)
_wan = generate_gold_efficacy_waning(_events, _people)   # vectorized
_ccr = generate_gold_cold_chain_risk(_inv, _sites)
_t1 = time.perf_counter()
_cur, _peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

_total = sum(len(d) for d in [_sites,_people,_appts,_inv,_events,_cov,_wan,_ccr])
_silver_t = _t_silver - _t0
_gold_t   = _t1 - _t_silver
_wall     = _t1 - _t0

print('='*58)
print('  INFERENCE EFFICIENCY BENCHMARK')
print('='*58)
print(f'  Silver layer (5 tables)  : {_silver_t:6.2f} s')
print(f'  Gold layer (3 tables)    : {_gold_t:6.2f} s  (waning vectorized)')
print(f'  Total wall time          : {_wall:6.2f} s')
print(f'  Peak memory              : {_peak/1e6:6.1f} MB')
print(f'  Total rows generated     : {_total:,}')
print(f'  Throughput               : {_total/_wall:,.0f} rows/second')
print('='*58)
print('\n  Complexity notes:')
print('   • Appointments  : O(people x vaccines x years)')
print('   • Waning (gold) : O(completed_doses x 60) — vectorized via')
print('                     NumPy broadcasting (~30x vs Python loop)')
print('   • Memory        : O(N) — no full cartesian materialization')
print('\n  Suitable as a production fail-safe: a fresh 800K-row dataset')
print('  regenerates in well under a typical CI test budget.')